In [146]:
import numpy as np
import networkx as nx

In [147]:
def construct_environment(exits, obstacles):
  environment = []
  for i in range(5):
    row = []
    for j in range(5):
      block = {"is_obstacle": False, "is_exit": False, "is_normal": False, "reward": 0, "direction": 0}
      if (i,j) in obstacles:
        block["is_obstacle"] = True
      elif (i,j) in exits.keys():
        block["is_exit"] = True
        block["reward"] = exits[(i,j)]
      else:
        block["is_normal"] = True
      row.append(block)
    environment.append(row)
  return environment

In [148]:
def get_children(environment, node, visited):
  i, j = node
  children = []
  for di, dj in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
    ni, nj = i + di, j + dj
    if 0 <= ni <= 4 and 0 <= nj <= 4 and environment[ni][nj]["is_normal"] and (ni, nj) not in visited:
      children.append((ni, nj))
  return children

def formula(intented, left, right, noise):
  return (1 - noise)*intented + (noise/2)*left + (noise/2)*right

def iterating_environment(environment, start, noise, live_in_reward):
  queue = []
  for _ in range(100):
    queue.extend(get_children(environment, start, queue))
    k = 0
    while(k < len(queue)):
      i, j = queue[k]
      k += 1
      queue.extend(get_children(environment, (i, j), queue))
      adj_reward = []
      for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        ni, nj = i + di, j + dj
        if 0 <= ni <= 4 and 0 <= nj <= 4:
          adj_reward.append(environment[ni][nj]["reward"])
        else:
          adj_reward.append(environment[i][j]["reward"])

      calculate_direction = []
      calculate_direction.append(formula(adj_reward[0], adj_reward[2], adj_reward[3], noise))
      calculate_direction.append(formula(adj_reward[1], adj_reward[2], adj_reward[2], noise))
      calculate_direction.append(formula(adj_reward[2], adj_reward[0], adj_reward[1], noise))
      calculate_direction.append(formula(adj_reward[3], adj_reward[0], adj_reward[1], noise))

      max_reward = max(calculate_direction)
      for direction in range(4):
        if calculate_direction[direction] == max_reward:
          environment[i][j]["direction"] = direction
          break
      environment[i][j]["reward"] = live_in_reward + 0.99 * max_reward

In [149]:
def get_arrow(i):
  arrow = ['↑     ', '↓     ', '←     ', '→     ']
  return arrow[i]

def print_env(env):
  for i in range(5):
      for j in range(5):
          if env[i][j]["is_obstacle"]:
              print("X    ", end=' ')
          else:
              print(f"{round(env[i][j]['reward'],2):<6}", end = '')
      print()

  print("============================")

  for i in range(5):
      for j in range(5):
          if env[i][j]["is_obstacle"]:
              print("□    ", end=' ')
          elif env[i][j]["is_exit"]:
              print(f"{round(env[i][j]['reward'],2):<6}", end = '')
          else:
              print(f"{get_arrow(env[i][j]['direction'])}", end = '')
      print()

In [150]:
def construct_graph(environment):
    G = nx.DiGraph()
    for i in range(5):
        for j in range(5):
            if not environment[i][j]["is_normal"]:
                continue
            direction = environment[i][j]["direction"]
            di, dj = [(-1, 0), (1, 0), (0, -1), (0, 1)][direction]
            ni, nj = i + di, j + dj
            if 0 <= ni <= 4 and 0 <= nj <= 4 and not environment[ni][nj]["is_obstacle"]:
                G.add_edge((i, j), (ni, nj), weight=environment[ni][nj]["reward"])
    return G

def find_total_reward(G, target, environment, max_steps):
    total_reward = 0

    for i in range(5):
        for j in range(5):
            start = (i, j)
            if not environment[i][j]["is_normal"]:
                continue
            try:
                paths = nx.all_simple_paths(G, source=start, target=target, cutoff=max_steps)
                paths = list(paths)
            except nx.NetworkXNoPath:
                continue

            for path in paths:
                total_reward += sum(environment[pi][pj]["reward"] for pi, pj in path)

    return total_reward

# R1

In [151]:
for i in np.arange(-0.3, -0.05, 0.05):
    env = construct_environment(
      {
          (4, 0): -10,
          (4, 1): -10,
          (4, 2): -10,
          (4, 3): -10,
          (4, 4): -10,
          (2, 2): 1,
          (2, 4): 10
      },
      [(2, 1), (2, 3), (0, 4)]
    )
    print(f"\033[1mr: {i:.2f}\033[0m")
    print("==========")
    iterating_environment(env, (2, 4), 0.1, i)

    G = construct_graph(env)
    total_reward = find_total_reward(G, (2, 4), env, max_steps=10)

    print_env(env)
    print("============================")
    print(f"Total Reward: {total_reward:.2f}")
    print()


r: -0.30
6.77  7.2   7.63  8.05  X     
6.5   6.93  7.72  8.52  9.45  
5.78  X     1     X     10    
5.34  4.6   6.05  7.63  9.46  
-10   -10   -10   -10   -10   
→     →     →     ↓     □     
→     →     →     →     ↓     
↑     □     1     □     10    
↑     →     →     →     ↑     
-10   -10   -10   -10   -10   
Total Reward: 595.61

r: -0.25
7.09  7.47  7.85  8.22  X     
6.77  7.14  7.88  8.63  9.51  
6.08  X     1     X     10    
5.68  4.77  6.19  7.73  9.51  
-10   -10   -10   -10   -10   
→     →     →     ↓     □     
→     →     →     →     ↓     
↑     □     1     □     10    
↑     →     →     →     ↑     
-10   -10   -10   -10   -10   
Total Reward: 604.02

r: -0.20
7.43  7.75  8.08  8.39  X     
7.14  7.46  8.04  8.75  9.58  
6.48  X     1     X     10    
6.12  4.95  6.33  7.83  9.57  
-10   -10   -10   -10   -10   
→     →     →     ↓     □     
↑     ↑     →     →     ↓     
↑     □     1     □     10    
↑     →     →     →     ↑     
-10   -10   -10   -10   -10   

### Ideal r for R1 would be r = -0.25

Reason for selection is provided in the doc

In [152]:
env = construct_environment(
      {
          (4, 0): -10,
          (4, 1): -10,
          (4, 2): -10,
          (4, 3): -10,
          (4, 4): -10,
          (2, 2): 1,
          (2, 4): 10
      },
      [(2, 1), (2, 3), (0, 4)]
    )
iterating_environment(env, (2, 4), 0.1, -0.25)
print_env(env)

7.09  7.47  7.85  8.22  X     
6.77  7.14  7.88  8.63  9.51  
6.08  X     1     X     10    
5.68  4.77  6.19  7.73  9.51  
-10   -10   -10   -10   -10   
→     →     →     ↓     □     
→     →     →     →     ↓     
↑     □     1     □     10    
↑     →     →     →     ↑     
-10   -10   -10   -10   -10   


# R2

In [153]:
for i in np.arange(-0.3, -0.05, 0.05):
    env = construct_environment(
      {
          (4,0): -10,
          (4,1): -10,
          (4,2): -10,
          (4,3): -10,
          (4,4): -10,
          (2,2): 1,
          (2,4): 10
      },
      [(2,1), (2,3), (1,3)]
    )
    print(f"\033[1mr: {i:.2f}\033[0m")
    print("==========")
    iterating_environment(env, (2, 4), 0.1, i)

    G = construct_graph(env)
    total_reward = find_total_reward(G, (2, 4), env, max_steps=10)

    print_env(env)
    print("============================")
    print(f"Total Reward: {total_reward:.2f}")
    print()


r: -0.30
5.96  6.38  6.81  7.27  8.09  
5.58  5.96  6.07  X     8.61  
4.92  X     1     X     10    
4.53  4.6   6.05  7.63  9.46  
-10   -10   -10   -10   -10   
→     →     →     →     ↓     
↑     ↑     ↑     □     ↓     
↑     □     1     □     10    
↑     →     →     →     ↑     
-10   -10   -10   -10   -10   
Total Reward: 628.24

r: -0.25
6.28  6.65  7.03  7.42  8.2   
5.95  6.28  6.32  X     8.66  
5.32  X     1     X     10    
4.97  4.77  6.19  7.73  9.51  
-10   -10   -10   -10   -10   
→     →     →     →     ↓     
↑     ↑     ↑     □     ↓     
↑     □     1     □     10    
↑     →     →     →     ↑     
-10   -10   -10   -10   -10   
Total Reward: 639.80

r: -0.20
6.6   6.92  7.24  7.58  8.31  
6.32  6.6   6.58  X     8.71  
5.72  X     1     X     10    
5.41  4.95  6.33  7.83  9.57  
-10   -10   -10   -10   -10   
→     →     →     →     ↓     
↑     ↑     ↑     □     ↓     
↑     □     1     □     10    
↑     →     →     →     ↑     
-10   -10   -10   -10   -10   

### Ideal r for R2 would be r = -0.1

Reason for selection is provided in the doc

In [154]:
env = construct_environment(
    {
        (4,0): -10,
        (4,1): -10,
        (4,2): -10,
        (4,3): -10,
        (4,4): -10,
        (2,2): 1,
        (2,4): 10
     },
    [(2,1), (2,3), (1,3)]
)
iterating_environment(env, (2, 4), 0.1, -0.1)
print_env(env)

7.25  7.45  7.66  7.89  8.53  
7.07  7.24  7.08  X     8.81  
6.52  X     1     X     10    
6.28  5.3   6.61  8.04  9.69  
-10   -10   -10   -10   -10   
→     →     →     →     ↓     
↑     ↑     ↑     □     ↓     
↑     □     1     □     10    
↑     →     →     →     ↑     
-10   -10   -10   -10   -10   
